# Maragoli mBART-50 Training
**Before running:** 

| Corpus | Train | Val | Test |
|--------|-------|-----|------|
| 1,642 pairs | 1,247 | 197 | 198 |

**Checkpoints save directly to Google Drive every epoch — session resets are safe.**

Expected time: ~60–90 minutes on T4 GPU.

In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Step 2: Install dependencies
!pip install transformers datasets sentencepiece sacrebleu accelerate scikit-learn -q

In [ ]:
# Step 3: Copy project from Drive to /content (skips models folder — fast)
import shutil, os

DRIVE_PROJECT = '/content/drive/MyDrive/MaragoliModel'
LOCAL_PROJECT = '/content/MaragoliModel'

if os.path.exists(LOCAL_PROJECT):
    shutil.rmtree(LOCAL_PROJECT)

def ignore_models(dir, contents):
    return ['models'] if 'models' in contents else []

shutil.copytree(DRIVE_PROJECT, LOCAL_PROJECT, ignore=ignore_models)
os.makedirs(f'{LOCAL_PROJECT}/models', exist_ok=True)

# Use %cd so all subsequent ! shell commands run from the right directory
%cd /content/MaragoliModel
print('Working directory:', os.getcwd())
print('Files:', os.listdir('.'))

In [ ]:
# Step 4: Verify GPU
import torch
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU name:', torch.cuda.get_device_name(0))
    print('GPU memory:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
else:
    print('WARNING: No GPU! Go to Runtime > Change runtime type > T4 GPU and restart')
    raise SystemExit('Stopping — please enable GPU first.')

In [ ]:
# Step 5: Verify data
import json
for split in ['train', 'val', 'test']:
    with open(f'data/{split}.json') as f:
        data = json.load(f)
    print(f'{split:5s}: {len(data)} pairs | sample: {data[0]["maragoli"][:50]}')

In [ ]:
# Step 6: Train (checkpoints saved directly to Drive every epoch)
import os
DRIVE_OUT = '/content/drive/MyDrive/MaragoliModel/models/mbart-maragoli'
os.makedirs(DRIVE_OUT, exist_ok=True)
print(f'Saving checkpoints to: {DRIVE_OUT}')
# {DRIVE_OUT} substitutes the Python variable into the shell command
!python scripts/02_train.py --epochs 20 --batch_size 8 --output_dir {DRIVE_OUT}


In [ ]:
# Step 7: Evaluate
# Updates MODEL_DIR inside 03_evaluate.py to point at the Drive checkpoint
import os, glob
DRIVE_OUT = '/content/drive/MyDrive/MaragoliModel/models/mbart-maragoli'

# Find the best checkpoint (highest-numbered or model.safetensors directly in DRIVE_OUT)
checkpoint = None
if os.path.exists(os.path.join(DRIVE_OUT, 'model.safetensors')):
    checkpoint = DRIVE_OUT
else:
    ckpts = sorted(glob.glob(f'{DRIVE_OUT}/checkpoint-*'))
    checkpoint = ckpts[-1] if ckpts else None

if checkpoint:
    print(f'Evaluating checkpoint: {checkpoint}')
    os.environ['EVAL_CHECKPOINT'] = checkpoint
    !python scripts/03_evaluate.py --model_dir "$EVAL_CHECKPOINT"
else:
    print('No checkpoint found — run Step 6 first')

In [ ]:
# Step 8: Confirm model is saved on Drive
import os
DRIVE_OUT = '/content/drive/MyDrive/MaragoliModel/models/mbart-maragoli'
if os.path.exists(DRIVE_OUT):
    files = [f for f in os.listdir(DRIVE_OUT) if os.path.isfile(os.path.join(DRIVE_OUT, f))]
    dirs  = [d for d in os.listdir(DRIVE_OUT) if os.path.isdir(os.path.join(DRIVE_OUT, d))]
    print(f"Drive path : {DRIVE_OUT}")
    print(f"Files      : {files}")
    print(f"Checkpoints: {dirs}")
    total = sum(os.path.getsize(os.path.join(DRIVE_OUT, f)) for f in files) / 1e6
    print(f"Total size : {total:.0f} MB")
else:
    print("ERROR: Model not found on Drive")

## After training
1. Run **Step 8** above to confirm the model files are on Drive
2. Download the model folder from Drive:
   
3. Place it at  on your local machine
4. Update  in  to point to that folder
5. Start the server: 